# EDA — Emotional Friction Detector
Explore the simulated dataset: frustration rate, delay distributions, event patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

sessions = pd.read_parquet('../data/raw/sessions.parquet')
events   = pd.read_parquet('../data/raw/events.parquet')
users    = pd.read_parquet('../data/raw/user_profiles.parquet')

print(f'Sessions: {len(sessions):,} | Events: {len(events):,} | Users: {len(users):,}')
print(f'Frustrated rate: {sessions.is_frustrated.mean():.1%}')

## 1. Delay distribution by context

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(axes,
    ['delay_minutes', 'base_eta_minutes', 'session_duration_seconds'],
    ['Delay (min)', 'Base ETA (min)', 'Session duration (s)']):
    ax.hist(sessions[col].clip(-10, 30), bins=50, edgecolor='none')
    ax.set_title(title)
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()

## 2. Frustration rate by trigger type

In [ ]:
sessions.frustration_trigger.value_counts().plot(kind='bar')
plt.title('Frustration triggers')
plt.ylabel('Count')
plt.tight_layout()

## 3. ETA refresh count: frustrated vs not

In [ ]:
fig, ax = plt.subplots()
for label, grp in sessions.groupby('is_frustrated'):
    ax.hist(grp.eta_refresh_count.clip(0, 15), bins=15, alpha=0.6, label=f'frustrated={label}')
ax.legend()
ax.set_xlabel('ETA refresh count')
ax.set_title('Refresh count distribution')
plt.tight_layout()

## 4. tap_interval_cv vs frustration label

In [ ]:
import sys; sys.path.insert(0, '..')
from features.sequence_features import extract_sequence_features

# Run on a sample for speed
sample_ids = sessions.sample(5000, random_state=42).session_id
sample_events = events[events.session_id.isin(sample_ids)]
seq_feats = extract_sequence_features(sample_events).reset_index()
merged = seq_feats.merge(sessions[['session_id','is_frustrated']], on='session_id')

for label, grp in merged.groupby('is_frustrated'):
    plt.hist(grp.tap_interval_cv.clip(0, 3), bins=40, alpha=0.6, label=f'frustrated={label}', density=True)
plt.xlabel('tap_interval_cv')
plt.legend()
plt.title('tap_interval_cv by frustration label (primary signal)')